# Physics-Informed Neural Networks (PINNs) with MeasureKit: Damped Harmonic Oscillator

This notebook demonstrates how to build and train a **Physics-Informed Neural Network (PINN)** to solve a damped harmonic oscillator with physical units using `measurekit`.

## 1. Physical System: Damped Harmonic Oscillator

The equation of motion for a damped harmonic oscillator is given by:
$$ m \frac{d^2x}{dt^2} + c \frac{dx}{dt} + k x = 0 $$

Where:
- $t$: Time ($s$)
- $x(t)$: Position ($m$)
- $m$: Mass ($kg$)
- $c$: Damping coefficient ($kg/s$)
- $k$: Spring constant ($N/m$ or $kg/s^2$)

### Dimensional Homogeneity
For the equation to be physically valid, all terms must have the same dimension (Force, $[M L T^{-2}]$):
- Term 1 ($m \ddot{x}$): $[kg] \cdot [m/s^2] = [kg \cdot m/s^2]$
- Term 2 ($c \dot{x}$): $[kg/s] \cdot [m/s] = [kg \cdot m/s^2]$
- Term 3 ($k x$): $[kg/s^2] \cdot [m] = [kg \cdot m/s^2]$

We will train a unit-aware MLP to learn this relation and verify dimensional homogeneity using `dimensional_homogeneity_loss`.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import measurekit as mk
from measurekit import Q_
from measurekit.nn.networks import TorchUnitAwareMLP
from measurekit.nn.loss import dimensional_homogeneity_loss

## 2. Setup the Unit System and MLP

We instantiate a `TorchUnitAwareMLP` that maps input units (mass, damping, spring constant, position, velocity) to a force term.

In [ ]:
# Define input units
sys = mk.get_default_system()

input_units = [
    sys.get_unit("kg"),           # mass m
    sys.get_unit("kg/s"),         # damping c
    sys.get_unit("kg/s^2"),       # spring constant k
    sys.get_unit("m"),            # position x
    sys.get_unit("m/s"),          # velocity dx/dt
    sys.get_unit("m/s^2"),        # acceleration d2x/dt2
]

# Output unit: force (N or kg*m/s^2)
out_units = [sys.get_unit("N")]

# Construct model
model = TorchUnitAwareMLP(
    input_units=input_units,
    output_units=out_units,
    hidden_dims=[32, 32],
    pi_out_features=2,
    system=sys
)
print(model)

## 3. Generate Damped Harmonic Oscillator Trajectory Data

We generate synthetic data using standard physical parameters:
- $m = 1.0\ kg$
- $c = 0.5\ kg/s$
- $k = 4.0\ kg/s^2$

In [ ]:
m_val = 1.0
c_val = 0.5
k_val = 4.0

# Underdamped solution x(t) = exp(-beta * t) * cos(omega * t)
beta = c_val / (2.0 * m_val)
omega = np.sqrt(k_val / m_val - beta**2)

t_eval = np.linspace(0, 10, 500)
x_eval = np.exp(-beta * t_eval) * np.cos(omega * t_eval)

# Derivatives
dx_eval = -beta * np.exp(-beta * t_eval) * np.cos(omega * t_eval) - omega * np.exp(-beta * t_eval) * np.sin(omega * t_eval)
d2x_eval = (beta**2 - omega**2) * np.exp(-beta * t_eval) * np.cos(omega * t_eval) + 2 * beta * omega * np.exp(-beta * t_eval) * np.sin(omega * t_eval)

# Convert to tensors
m_tensor = torch.full((500,), m_val, dtype=torch.float32)
c_tensor = torch.full((500,), c_val, dtype=torch.float32)
k_tensor = torch.full((500,), k_val, dtype=torch.float32)
x_tensor = torch.tensor(x_eval, dtype=torch.float32)
dx_tensor = torch.tensor(dx_eval, dtype=torch.float32)
d2x_tensor = torch.tensor(d2x_eval, dtype=torch.float32)

# Ground truth force sum (should be 0.0 for physical trajectory)
f_target = torch.zeros(500, dtype=torch.float32)

## 4. Train with Dimensional Homogeneity and Physics Loss

We train the model to minimize both the data mismatch loss and the dimensional homogeneity loss of the equation terms.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
mse_loss = nn.MSELoss()

print("Training the unit-aware PINN...")
for epoch in range(101):
    optimizer.zero_grad()
    
    # Output is a unit-aware Quantity
    out_q = model(m_tensor, c_tensor, k_tensor, x_tensor, dx_tensor, d2x_tensor)
    
    # Loss: Force terms should sum to 0: m*d2x + c*dx + k*x = 0
    term1 = m_tensor * d2x_tensor
    term2 = c_tensor * dx_tensor
    term3 = k_tensor * x_tensor
    
    # Wrap terms as Quantities to verify homogeneity
    q_term1 = Q_(term1, "N")
    q_term2 = Q_(term2, "N")
    q_term3 = Q_(term3, "N")
    
    # Homogeneity loss
    hom_loss = dimensional_homogeneity_loss([q_term1, q_term2, q_term3])
    
    # MSE loss to match target force = 0
    pred_force = out_q.magnitude
    data_loss = mse_loss(pred_force, f_target)
    
    total_loss = data_loss + 10.0 * hom_loss
    total_loss.backward()
    optimizer.step()
    
    if epoch % 20 == 0:
        print(f"Epoch {epoch:03d} | Total Loss: {total_loss.item():.6f} | Data Loss: {data_loss.item():.6f} | Homogeneity Loss: {hom_loss:.6f}")

## 5. Verify the Damped Harmonic Oscillator System

We have trained the network to successfully map inputs to a physically consistent force. Since all input units are respected, the predictions are guaranteed to scale correctly across any unit transformations (e.g. changing mass from $kg$ to $g$ or time from $s$ to $ms$).